In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import joblib


In [4]:
#Load the dataset
df = pd.read_csv('../data~/transactions.csv')
print(f"Dataset Shape: {df.shape}")

# Preprocessing
df_clean = df.drop(columns=['nameOrig', 'nameDest', 'isFlaggedFraud'])
df_clean = pd.get_dummies(df_clean, columns=['type'], drop_first=True)

X = df_clean.drop(columns=['isFraud'])
y = df_clean['isFraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Dataset Shape: (6362620, 11)


In [8]:
# 3. Train Model
model = RandomForestClassifier(
    n_estimators=100, 
    max_depth=12, 
    class_weight='balanced', 
    random_state=42, 
    n_jobs=-1
)
model.fit(X_train_scaled, y_train)

print("7. Evaluating model performance...")
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print("\n--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

roc_auc = roc_auc_score(y_test, y_prob)
print(f"ROC-AUC Score: {roc_auc:.4f}")

print("8. Saving model artifacts...")
joblib.dump(model, '../models/fraud_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
print("Success! Model and scaler successfully saved inside the 'models/' folder.")

7. Evaluating model performance...

--- Confusion Matrix ---
[[1259374   11507]
 [     24    1619]]

--- Classification Report ---
              precision    recall  f1-score   support

           0       1.00      0.99      1.00   1270881
           1       0.12      0.99      0.22      1643

    accuracy                           0.99   1272524
   macro avg       0.56      0.99      0.61   1272524
weighted avg       1.00      0.99      0.99   1272524

ROC-AUC Score: 0.9992
8. Saving model artifacts...
Success! Model and scaler successfully saved inside the 'models/' folder.
